# Artwork Eval — Drive + Row-level Checkpoint
Model: `llava-hf/llama3-llava-next-8b-hf` | Press: `ExpectedAttentionPress`

Results + checkpoint saved to **Google Drive** — safe to resume after disconnect.

> After disconnect: re-run **Step 1.5 (Drive) → Step 3 (model) → Step 6 (inference)**.


## Step 0 — Set runtime to GPU (T4) before running.

## Step 0.5 — GitHub PAT

In [ ]:
import getpass, os
manual = getpass.getpass("GitHub Classic PAT (leave empty to use Colab Secret): ").strip()
if manual:
    gh_token = manual
else:
    try:
        from google.colab import userdata
        gh_token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        gh_token = os.environ.get("GITHUB_TOKEN", "").strip()
print("Token loaded:", bool(gh_token))

## Step 1 — Clone repos & install deps

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL    = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"
BENCH_DIR = Path("/content/kv-compression-benchmark")
CE_DIR    = Path("/content/CompressionExperiments")

def clone(url, out, token, name):
    r = subprocess.run(["git","clone","--depth","1",url,str(out)], capture_output=True, text=True)
    if r.returncode == 0: print(name, "public clone OK"); return True
    print(name, "public clone failed"); print((r.stderr or "")[:300])
    if not token: return False
    r2 = subprocess.run(["git","clone","--depth","1",url.replace("https://github.com/",f"https://{token}@github.com/"),str(out)], capture_output=True, text=True)
    if r2.returncode == 0: print(name, "token clone OK"); return True
    print(name, "token clone failed"); print((r2.stderr or "")[:300]); return False

_tok = (globals().get("gh_token") or "").strip() or None
os.chdir("/content")
for p in [BENCH_DIR, CE_DIR]:
    if p.exists(): shutil.rmtree(p)
bench_ok = clone(BENCH_URL, BENCH_DIR, _tok, "BENCH")
ce_ok    = clone(CE_URL,    CE_DIR,    _tok, "CE")
if not bench_ok: raise RuntimeError("BENCH repo required for patches + evaluation.")
print("Clone done.")

In [ ]:
import subprocess, sys
from pathlib import Path
_ce = Path("/content/CompressionExperiments")
# Submodule update: optional, non-fatal (CE submodule may require auth)
if _ce.is_dir():
    _r = subprocess.run(["git","submodule","update","--init","--recursive"],
                        capture_output=True, text=True, cwd=str(_ce))
    if _r.returncode != 0:
        print("WARN: submodule update failed (non-fatal):", (_r.stderr or "")[:200])
    else:
        print("Submodule update OK.")
subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","pip"])
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "transformers>=5.0","accelerate","bitsandbytes","pandas",
    "pyyaml","scikit-learn","matplotlib","tqdm","pillow"])
# Install kvpress from CE submodule if available, else from PyPI
_kvp = _ce / "kvpress"
if _kvp.is_dir():
    subprocess.check_call([sys.executable,"-m","pip","install","-q","-e",str(_kvp)])
    print("kvpress: installed from CE submodule")
else:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","kvpress"])
    print("kvpress: installed from PyPI")
print("Install complete.")

## Step 1.5 — Mount Google Drive
Results + checkpoint are persisted here. Re-run this cell after every disconnect.

In [ ]:
from google.colab import drive
import os
from pathlib import Path
drive.mount("/content/drive", force_remount=False)
DRIVE_OUT        = "/content/drive/MyDrive/kv-compression-benchmark/ce_artwork_results"
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH  = Path(DRIVE_OUT) / "checkpoint.csv"
RESULTS_ROOT     = Path(DRIVE_OUT) / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("DRIVE_OUT   :", DRIVE_OUT)
print("CHECKPOINT  :", CHECKPOINT_PATH)

## Step 2 — Prepare artwork dataset
Priority: CE built-in images → BENCH repo images → download from Wikimedia URLs.

In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd

BENCH_DIR = Path("/content/kv-compression-benchmark")
CE_DIR    = Path("/content/CompressionExperiments")
CE_ART    = CE_DIR / "experiment_manager" / "datasets" / "artwork"
IMAGES_DIR = CE_ART / "images"
DATASET_PATH = CE_ART / "paintings.csv"

# 1) CSV check
if not DATASET_PATH.is_file():
    src_csv = BENCH_DIR / "paintings.csv"
    if src_csv.is_file():
        shutil.copy2(src_csv, DATASET_PATH)
        print("CSV: copied from BENCH repo")
    else:
        raise FileNotFoundError(f"paintings.csv not found at {DATASET_PATH} or {src_csv}")
else:
    print("CSV: found in CE repo")

# 2) Images check
if not IMAGES_DIR.is_dir():
    print(f"Error: Images directory not found at {IMAGES_DIR}")
    # Debug: show what is in the artwork folder
    if CE_ART.is_dir():
        print(f"Contents of {CE_ART}: {os.listdir(CE_ART)}")
    raise FileNotFoundError(f"Images directory missing: {IMAGES_DIR}")

img_count = len(list(IMAGES_DIR.glob("*")))
if img_count == 0:
    print(f"Error: Images directory is empty: {IMAGES_DIR}")
    raise FileNotFoundError("Images directory is empty.")

print(f"Images ready at: {IMAGES_DIR}")
print(f"Image count: {img_count}")


## Step 3 — Load model + kvpress patches
Re-run after disconnect together with Step 1.5.

In [ ]:
import sys, importlib.util, torch, transformers
from pathlib import Path
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
print("transformers:", transformers.__version__)

MODEL_ID  = "llava-hf/llama3-llava-next-8b-hf"
MODEL_TAG = MODEL_ID.split("/")[-1]
dtype     = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb_cfg   = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=True)

print(f"Loading {MODEL_ID} ...")
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=dtype, device_map="auto", quantization_config=bnb_cfg)

_patch_py = Path("/content/kv-compression-benchmark/benchmarks/artwork_eval/llava_kvpress_patch.py")
if not _patch_py.is_file():
    raise FileNotFoundError(f"Patch not found: {_patch_py}. Ensure BENCH repo is cloned (Step 1).")
_spec = importlib.util.spec_from_file_location("llava_kvpress_patch", _patch_py)
_mod  = importlib.util.module_from_spec(_spec)
sys.modules["llava_kvpress_patch"] = _mod
_spec.loader.exec_module(_mod)
_mod.apply_kvpress_compatibility_patches(model)
print("Model ready:", MODEL_ID)

## Step 4 — Configuration

In [ ]:
from pathlib import Path
from kvpress import ExpectedAttentionPress, KVzipPress, FinchPress

ARTWORK_EVAL_ROOT  = Path("/content/kv-compression-benchmark/benchmarks/artwork_eval")
IMAGE_QUERIES_YAML = ARTWORK_EVAL_ROOT / "configs" / "image_queries.yaml"
EVAL_CONFIG_YAML   = ARTWORK_EVAL_ROOT / "evaluation" / "evaluation_config.yaml"
DATASET_NAME       = "artwork"

MAX_ROWS           = 10      # set 0 for full dataset
MAX_NEW_TOKENS     = 50
COMPRESSION_RATIOS = [0.4, 0.8, 0.95]
PRESS_NAMES        = ["ExpectedAttentionPress"]
# PRESS_NAMES      += ["KVzipPress"]  # uncomment if needed

PRESS_REGISTRY = {
    "ExpectedAttentionPress": ExpectedAttentionPress,
    "KVzipPress": KVzipPress,
    "FinchPress": FinchPress,
}

_EXTRACT_SUFFIX = (" (be concise, no explanation, no introductory text, just the answer,"
                   " output datatype: STRING, do not repeat the datatype in the answer) ?")

def build_answer_prefix(question: str, is_boolean: bool) -> str:
    if is_boolean:
        return (f"Answer the following question based on the image with '1' or '0'."
                f" Do not add any other comments. {question}\nAnswer: ")
    return (f"Answer the following question based on the image."
            f" Do not add any other comments. {question + _EXTRACT_SUFFIX}\nAnswer: ")

print("Config ready. MAX_ROWS:", MAX_ROWS, "| RATIOS:", COMPRESSION_RATIOS)

## Step 5 — Load dataset & queries

In [ ]:
from urllib.parse import unquote
import yaml, pandas as pd

with open(IMAGE_QUERIES_YAML, encoding="utf-8") as f:
    _y = yaml.safe_load(f)
art_cfg         = _y["artwork"]
filter_queries  = list(art_cfg["filter_queries"])
extract_queries = list(art_cfg["extract_queries"])
img_url_col     = art_cfg.get("image_url_column", "image_url")

df = pd.read_csv(DATASET_PATH)
df = df[df[img_url_col].notna()].copy().reset_index(drop=True)
if MAX_ROWS and MAX_ROWS > 0:
    df = df.head(int(MAX_ROWS))
df.insert(0, "record_id", range(len(df)))

def resolve_image(url: str) -> str:
    tail = url.split("/")[-1].split("?")[0]
    p = IMAGES_DIR / unquote(tail)
    return str(p) if p.is_file() else str(IMAGES_DIR / tail)

df["image_path"] = df[img_url_col].apply(resolve_image)
queries_plan = [(q, True) for q in filter_queries] + [(q, False) for q in extract_queries]
print("Rows:", len(df), "| Queries per record:", len(queries_plan),
      "| Total inferences:", len(df) * len(queries_plan) * len(PRESS_NAMES) * len(COMPRESSION_RATIOS))

## Step 6 — Checkpointed inference
Safe to re-run: already-completed (press, ratio, record_id, query) tuples are skipped.

In [ ]:
import csv as _csv, gc, os, time, torch
from pathlib import Path
from PIL import Image

_tok = getattr(processor, "tokenizer", processor)

def run_generate_vision(image_path, answer_prefix, press_name, ratio):
    sz = os.path.getsize(image_path)
    if sz < 512:
        raise OSError(f"Image too small ({sz} B): {image_path}")
    press = PRESS_REGISTRY[press_name](compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv  = [{"role":"user","content":[{"type":"image"},{"type":"text","text":answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": MAX_NEW_TOKENS}
    if getattr(_tok, "pad_token_id", None) is not None:
        gen_kw["pad_token_id"] = _tok.pad_token_id
    with torch.no_grad(), press(model):
        out = model.generate(**inputs, **gen_kw)
    inp_len = inputs["input_ids"].shape[1]
    return _tok.decode(out[0][inp_len:], skip_special_tokens=True).strip()

# --- Checkpoint helpers ---
CK_FIELDS = ["press","ratio","record_id","query","answer","error"]

def load_done(path):
    if not path.is_file(): return set()
    ck = pd.read_csv(path)
    done = set()
    for _, r in ck.iterrows():
        if str(r.get("error","")).strip().lower() in ("","nan"):
            done.add((str(r["press"]), str(float(r["ratio"])), int(r["record_id"]), str(r["query"])))
    return done

def append_ck(path, row):
    newfile = not path.is_file()
    with path.open("a", newline="", encoding="utf-8") as f:
        w = _csv.DictWriter(f, fieldnames=CK_FIELDS, extrasaction="ignore")
        if newfile: w.writeheader()
        w.writerow({k: row.get(k,"") for k in CK_FIELDS})
        f.flush(); os.fsync(f.fileno())

# --- Run ---
done_keys = load_done(CHECKPOINT_PATH)
print(f"Resuming: {len(done_keys)} already done.")
total = len(df) * len(queries_plan) * len(PRESS_NAMES) * len(COMPRESSION_RATIOS)
n_done = 0

for pname in PRESS_NAMES:
    for ratio in COMPRESSION_RATIOS:
        for _, row in df.iterrows():
            rid = int(row["record_id"])
            ip  = row["image_path"]
            if not os.path.isfile(ip):
                print(f"[skip] Missing image for record {rid}: {ip}"); continue
            for qtext, is_bool in queries_plan:
                key = (pname, str(float(ratio)), rid, qtext)
                if key in done_keys:
                    n_done += 1; continue
                ap = build_answer_prefix(qtext, is_bool)
                err = ans = ""
                try:
                    ans = run_generate_vision(ip, ap, pname, float(ratio))
                except Exception as e:
                    err = str(e)[:500]
                    print(f"[err] rid={rid} {pname} r={ratio}: {err[:80]}")
                append_ck(CHECKPOINT_PATH, {"press":pname,"ratio":ratio,"record_id":rid,"query":qtext,"answer":ans,"error":err})
                if not err:
                    done_keys.add(key)
                n_done += 1
                if n_done % 10 == 0:
                    print(f"Progress: {n_done}/{total}")
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Inference done. {n_done}/{total} total. Checkpoint: {CHECKPOINT_PATH}")

## Step 7 — Reconstruct CE-style results tree from checkpoint
Re-runnable without re-inference.

In [ ]:
import pandas as pd
from pathlib import Path

ck = pd.read_csv(CHECKPOINT_PATH)
ck_ok = ck[ck["error"].fillna("").astype(str).str.strip().isin(["","nan"])].copy()

for (pname, ratio), grp in ck_ok.groupby(["press","ratio"]):
    ratio_tag = f"{float(ratio):.2f}"
    out_dir = RESULTS_ROOT / DATASET_NAME / MODEL_TAG / pname / ratio_tag
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "results.csv"
    grp[["record_id","query","press","ratio","answer"]].to_csv(out_path, index=False)
    print("Saved", out_path)
print("Results tree ready.")

## Step 8 — Evaluate (P/R/F1)

In [ ]:
import sys
from pathlib import Path
_ae = Path("/content/kv-compression-benchmark/benchmarks/artwork_eval")
if str(_ae) not in sys.path: sys.path.insert(0, str(_ae))
from evaluation.evaluator import EvaluationManager

mgr   = EvaluationManager(config_path=str(EVAL_CONFIG_YAML), results_dir=str(RESULTS_ROOT))
ev_df = mgr.evaluate_all()
if ev_df.empty:
    print("No results to evaluate. Check Steps 6-7.")
else:
    gcols = ["dataset","model_tag","press","ratio"]
    mcols = ["precision","recall","f1"]
    for label, sub in [("filter", ev_df[ev_df["query_type"]=="filter"]),
                       ("extract", ev_df[ev_df["query_type"]=="extract"]),
                       ("all", ev_df)]:
        print(f"\n=== {label} ===")
        display(sub.groupby(gcols)[mcols].mean().round(4))
    summ = Path(DRIVE_OUT) / "evaluation_results.csv"
    ev_df.to_csv(summ, index=False)
    print("Saved:", summ)